# Cross-domain contribution analysis

Phan tich dong gop cua shared layer va cross-domain reasoning tu artifact summary da export. Notebook nay tap trung vao domain pair nhu enterprise->tax va enterprise->labor.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SUMMARY_PATH = ROOT / 'data' / 'processed' / 'evaluation' / 'qa_eval_cross_domain_summary.json'
SUMMARY_PATH

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
summary.keys()

In [ ]:
overview = pd.DataFrame([
    {
        'rows_total': summary.get('rows_total', 0),
        'jump_attempted_total': summary.get('cross_domain_jumps_attempted_total', 0),
        'jump_success_total': summary.get('cross_domain_jumps_success_total', 0),
        'jump_blocked_total': summary.get('cross_domain_jumps_blocked_total', 0),
        'jump_success_rate': summary.get('cross_domain_jump_success_rate', 0.0),
        'shared_layer_activation_rate': (summary.get('contribution_metrics') or {}).get('shared_layer_activation_rate', 0.0),
        'bridge_usage_rate': (summary.get('contribution_metrics') or {}).get('bridge_usage_rate', 0.0),
    }
])
overview

In [ ]:
pair_freq = pd.DataFrame(
    [
        {'domain_pair': pair, 'count': count}
        for pair, count in (summary.get('domain_pair_transition_frequency') or {}).items()
    ]
).sort_values('count', ascending=False)
pair_freq

In [ ]:
focus_pairs = ['enterprise->tax', 'enterprise->labor']
focus_df = pair_freq[pair_freq['domain_pair'].isin(focus_pairs)].copy()
if focus_df.empty:
    focus_df = pd.DataFrame({'domain_pair': focus_pairs, 'count': [0, 0]})
focus_df

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 4.8))
colors = ['#15616d', '#ff7d00']
ax.bar(focus_df['domain_pair'], focus_df['count'], color=colors[: len(focus_df)])
ax.set_title('Cross-domain contribution by focal domain pair')
ax.set_xlabel('Domain pair')
ax.set_ylabel('Successful transitions in summary')
for idx, value in enumerate(focus_df['count']):
    ax.text(idx, value + 0.02, str(value), ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
bridge_pair_freq = pd.DataFrame(
    [
        {
            'domain_pair_bridge': pair_bridge,
            'count': count,
            'domain_pair': pair_bridge.split('::')[0],
            'bridge_rule': pair_bridge.split('::')[1] if '::' in pair_bridge else '',
        }
        for pair_bridge, count in (summary.get('domain_pair_bridge_frequency') or {}).items()
    ]
).sort_values(['domain_pair', 'count'], ascending=[True, False])
bridge_pair_freq

## Cach su dung

1. Chay `scripts/export_qa_eval_logs.py --input ... --summary-json data/processed/evaluation/qa_eval_cross_domain_summary.json`.
2. Mo notebook nay va chay tu Cell 2.
3. Neu can report rieng, dung them `scripts/aggregate_cross_domain_metrics.py --input data/processed/evaluation/qa_eval_logs.jsonl`.

In [ ]:
figures_dir = ROOT / 'paper' / 'figures'

processed_figures_dir = ROOT / 'data' / 'processed' / 'evaluation' / 'figures'

figures_dir.mkdir(parents=True, exist_ok=True)

processed_figures_dir.mkdir(parents=True, exist_ok=True)



export_path_paper = figures_dir / 'cross_domain_contribution_pairs.png'

export_path_processed = processed_figures_dir / 'cross_domain_contribution_pairs.png'



fig, ax = plt.subplots(figsize=(8, 4.8))

colors = ['#15616d', '#ff7d00']

ax.bar(focus_df['domain_pair'], focus_df['count'], color=colors[: len(focus_df)])

ax.set_title('Cross-domain contribution by focal domain pair')

ax.set_xlabel('Domain pair')

ax.set_ylabel('Successful transitions in summary')

for idx, value in enumerate(focus_df['count']):

    ax.text(idx, value + 0.02, str(value), ha='center', va='bottom')

fig.tight_layout()

fig.savefig(export_path_paper, dpi=220, bbox_inches='tight')

fig.savefig(export_path_processed, dpi=220, bbox_inches='tight')

plt.show()



print(f'paper figure: {export_path_paper}')

print(f'processed figure: {export_path_processed}')
